In [ ]:

from langchain_core.runnables import RunnableLambda 
from operator import itemgetter 
# ✅ Window Memory — keep only last K messages 
# We implement this by trimming the history before passing it 

from langchain_core.chat_history import InMemoryChatMessageHistory 
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, AIMessage , trim_messages 
from dotenv import load_dotenv

import os
 
load_dotenv(override=True)
 
api_key = os.getenv("ANTHROPIC_API_KEY")
if api_key is None:
    raise ValueError("ANTHROPIC_API_KEY not found in .env file")
base_url=os.getenv("BASE_URL")
model_name=os.getenv("MODEL_NAME")
 
llm = ChatAnthropic(model=model_name,base_url=base_url,anthropic_api_key=api_key, max_tokens=200)

WINDOW_SIZE = 4   # keep last 4 messages (2 human + 2 AI) 
 
prompt = ChatPromptTemplate.from_messages([ 
    ('system', 'You are a helpful assistant. Be concise.'), 
    MessagesPlaceholder(variable_name='chat_history'), 
    ('human', '{input}'), 
]) 
 
# Trim history to last WINDOW_SIZE messages before passing to prompt 
def trim_history(inputs): 
    inputs['chat_history'] = inputs['chat_history'][-WINDOW_SIZE:] 
    return inputs 
 
windowed_chain = RunnableLambda(trim_history) | prompt | llm | StrOutputParser() 
 
store_w = {} 
 
def get_session_history_w(session_id: str): 
    if session_id not in store_w: 
        store_w[session_id] = InMemoryChatMessageHistory() 
    return store_w[session_id] 
 
windowed_chain_with_memory = RunnableWithMessageHistory( 
    windowed_chain, 
    get_session_history_w, 
    input_messages_key='input', 
    history_messages_key='chat_history', 
) 
 
config_w = {'configurable': {'session_id': 'window_session'}} 
 
r1 = windowed_chain_with_memory.invoke({'input': 'My name is Ali.'}, config=config_w) 
print('Turn 1:', r1) 
 
r2 = windowed_chain_with_memory.invoke({'input': 'I am from India.'}, config=config_w) 
print('Turn 2:', r2) 
 
r3 = windowed_chain_with_memory.invoke({'input': 'I love Python.'}, config=config_w) 
print('Turn 3:', r3) 
 
# Turn 4: Window only shows last 4 messages — earliest context may be lost 
r4 = windowed_chain_with_memory.invoke({'input': 'What is my name and where am I from?'}, config=config_w) 
print('Turn 4:', r4)   # May forget 'Ali' depending on window size 

/Software/langenv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3701: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)
/tmp/ipykernel_119038/3318602136.py:45: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  store_w[session_id] = InMemoryChatMessageHistory()


Turn 1: Hi Ali! How can I help you today?
Turn 2: That's great, Ali! India is a wonderful country with rich culture and diversity. How can I assist you today?
Turn 3: That's awesome, Ali! Python is a great programming language. It's versatile, easy to learn, and widely used in areas like web development, data science, machine learning, and automation. Do you work on any specific projects or fields using Python?
Turn 4: Based on our conversation, your name is **Ali** and you are from **India**. 😊
